In [ ]:
#Para embeddings model

import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text

#Bibliotecas para análise de Clusters
import umap
from hyperopt import fmin, tpe, hp, STATUS_OK, space_eval, Trials
from functools import partial
import hdbscan

In [ ]:
#com busca de hiper para Umap:
def generate_clusters(message_embeddings,
                      n_neighbors,
                      n_components,
                      min_cluster_size,
                      min_samples = None,
                      random_state = None):
    """
    Returns HDBSCAN objects after first performing dimensionality reduction using UMAP

    Arguments:
        message_embeddings: embeddings to use
        n_neighbors: int, UMAP hyperparameter n_neighbors
        n_components: int, UMAP hyperparameter n_components
        min_cluster_size: int, HDBSCAN hyperparameter min_cluster_size
        min_samples: int, HDBSCAN hyperparameter min_samples
        random_state: int, random seed

    Returns:
        clusters: HDBSCAN object of clusters
    """

    umap_embeddings = (umap.UMAP(n_neighbors = n_neighbors,
                                n_components = n_components,
                                metric = 'cosine',
                                random_state=random_state)
                            .fit_transform(message_embeddings))

    clusters = hdbscan.HDBSCAN(min_cluster_size = min_cluster_size,
                               min_samples = min_samples,
                               metric='euclidean',
                               gen_min_span_tree=True,
                               cluster_selection_method='eom').fit(umap_embeddings)

    return clusters

def score_clusters(clusters, prob_threshold = 0.05):
    """
    Returns the label count and cost of a given clustering

    Arguments:
        clusters: HDBSCAN clustering object
        prob_threshold: float, probability threshold to use for deciding
                        what cluster labels are considered low confidence

    Returns:
        label_count: int, number of unique cluster labels, including noise
        cost: float, fraction of data points whose cluster assignment has
              a probability below cutoff threshold
    """

    cluster_labels = clusters.labels_
    label_count = len(np.unique(cluster_labels))
    total_num = len(clusters.labels_)
    cost = (np.count_nonzero(clusters.probabilities_ < prob_threshold)/total_num)

    return label_count, cost

def plot_clusters(embeddings, clusters, n_neighbors=15, min_dist=0.1):
    """
    Reduce dimensionality of best clusters and plot in 2D

    Arguments:
        embeddings: embeddings to use
        clusteres: HDBSCAN object of clusters
        n_neighbors: float, UMAP hyperparameter n_neighbors
        min_dist: float, UMAP hyperparameter min_dist for effective
                  minimum distance between embedded points

    """
    umap_data = umap.UMAP(n_neighbors=n_neighbors,
                          n_components=2,
                          min_dist = min_dist,
                          #metric='cosine',
                          random_state=42).fit_transform(embeddings)

    point_size = 100.0 / np.sqrt(embeddings.shape[0])

    result = pd.DataFrame(umap_data, columns=['x', 'y'])
    result['labels'] = clusters.labels_

    fig, ax = plt.subplots(figsize=(14, 8))
    outliers = result[result.labels == -1]
    clustered = result[result.labels != -1]
    plt.scatter(outliers.x, outliers.y, color = 'lightgrey', s=point_size)
    plt.scatter(clustered.x, clustered.y, c=clustered.labels, s=point_size, cmap='jet')
    plt.colorbar()
    plt.show()

def bayesian_search(embeddings, space, label_lower, label_upper, max_evals=100):
    """
    Perform bayesian search on hyperparameter space using hyperopt

    Arguments:
        embeddings: embeddings to use
        space: dict, contains keys for 'n_neighbors', 'n_components',
               'min_cluster_size', and 'random_state' and
               values that use built-in hyperopt functions to define
               search spaces for each
        label_lower: int, lower end of range of number of expected clusters
        label_upper: int, upper end of range of number of expected clusters
        max_evals: int, maximum number of parameter combinations to try

    Saves the following to instance variables:
        best_params: dict, contains keys for 'n_neighbors', 'n_components',
               'min_cluster_size', 'min_samples', and 'random_state' and
               values associated with lowest cost scenario tested
        best_clusters: HDBSCAN object associated with lowest cost scenario
                       tested
        trials: hyperopt trials object for search

        """

    trials = Trials()
    fmin_objective = partial(objective,
                             embeddings=embeddings,
                             label_lower=label_lower,
                             label_upper=label_upper)

    best = fmin(fmin_objective,
                space = space,
                algo=tpe.suggest,
                max_evals=max_evals,
                trials=trials)

    best_params = space_eval(space, best)
    print ('best:')
    print (best_params)
    print (f"label count: {trials.best_trial['result']['label_count']}")

    best_clusters = generate_clusters(embeddings,
                                      n_neighbors = best_params['n_neighbors'],
                                      n_components = best_params['n_components'],
                                      min_cluster_size = best_params['min_cluster_size'],
                                      min_samples = best_params['min_samples'],
                                      random_state = best_params['random_state'])

    return best_params, best_clusters, trials

def objective(params, embeddings, label_lower, label_upper):
    """
    Objective function for hyperopt to minimize

    Arguments:
        params: dict, contains keys for 'n_neighbors', 'n_components',
               'min_cluster_size', 'random_state' and
               their values to use for evaluation
        embeddings: embeddings to use
        label_lower: int, lower end of range of number of expected clusters
        label_upper: int, upper end of range of number of expected clusters

    Returns:
        loss: cost function result incorporating penalties for falling
              outside desired range for number of clusters
        label_count: int, number of unique cluster labels, including noise
        status: string, hypoeropt status

        """

    clusters = generate_clusters(embeddings,
                                 n_neighbors = params['n_neighbors'],
                                 n_components = params['n_components'],
                                 min_cluster_size = params['min_cluster_size'],
                                 random_state = params['random_state'])

    label_count, cost = score_clusters(clusters, prob_threshold = 0.05)

    #15% penalty on the cost function if outside the desired range of groups
    if (label_count < label_lower) | (label_count > label_upper):
        penalty = 0.15
    else:
        penalty = 0

    loss = cost + penalty

    return {'loss': loss, 'label_count': label_count, 'status': STATUS_OK}

In [ ]:
module_url = "https://tfhub.dev/google/universal-sentence-encoder-multilingual/3"
model_use = hub.load(module_url)
print(f"module {module_url} loaded")
    
todas_transcricoes = list(df['texto_processado_embeddings'])
embeddings_transcricoes = model_use(todas_transcricoes)
print(len(embeddings_transcricoes))

#Definindo o espaço para busca de hiperparâmetros
hspace = {
    "n_neighbors": hp.choice('n_neighbors', range(2, 50)),
    "n_components": hp.choice('n_components', range(2, 50)),
    "min_cluster_size": hp.choice('min_cluster_size', range(2, 12)),
    "min_samples": hp.choice('min_samples', range(2, 12)),
    "random_state": 42
}

label_lower = 20
label_upper = 80
max_evals = 50

best_params_use, best_clusters_use, trials_use = bayesian_search(embeddings_transcricoes,
                                                                 space=hspace,
                                                                 label_lower=label_lower,
                                                                 label_upper=label_upper,
                                                                 max_evals=max_evals)

#Mostrando os melhores hiperparâmetros
#print(best_params_use)

In [ ]:
clusters = list(np.unique(best_clusters_use.labels_))
len(clusters)

In [ ]:

def plot_clusters(embeddings, clusters, n_neighbors=15, min_dist=0.1):
    """
    Reduce dimensionality of best clusters and plot in 2D

    Arguments:
        embeddings: embeddings to use
        clusters: HDBSCAN object of clusters
        n_neighbors: float, UMAP hyperparameter n_neighbors
        min_dist: float, UMAP hyperparameter min_dist for effective
                  minimum distance between embedded points

    """
    umap_data = umap.UMAP(n_neighbors=n_neighbors,
                          n_components=2,
                          min_dist=min_dist,
                          #metric='cosine',
                          random_state=42).fit_transform(embeddings)

    point_size = 100.0 / np.sqrt(embeddings.shape[0])

    result = pd.DataFrame(umap_data, columns=['x', 'y'])
    result['labels'] = clusters.labels_

    fig, ax = plt.subplots(figsize=(14, 8))
    outliers = result[result.labels == -1]
    clustered = result[result.labels != -1]

    scatter_outliers = ax.scatter(outliers.x, outliers.y, color='lightgrey', s=point_size)
    scatter_clustered = ax.scatter(clustered.x, clustered.y, c=clustered.labels, s=point_size, cmap='jet')
    plt.colorbar(scatter_clustered, ax=ax)

    # Adicionando números de cluster aos pontos
    for i, txt in enumerate(clustered.labels):
        ax.annotate(txt, (clustered.x.iloc[i], clustered.y.iloc[i]), fontsize=8, ha='center', va='center')

    plt.show()

In [ ]:
#Função para Gráfico Interativo

import plotly.express as px

def plot_clusters_interactive(embeddings, clusters, n_neighbors=15, min_dist=0.1):
    """
    Reduce dimensionality of best clusters and create an interactive 2D plot

    Arguments:
        embeddings: embeddings to use
        clusters: HDBSCAN object of clusters
        n_neighbors: float, UMAP hyperparameter n_neighbors
        min_dist: float, UMAP hyperparameter min_dist for effective
                  minimum distance between embedded points

    """
    umap_data = umap.UMAP(n_neighbors=n_neighbors,
                          n_components=2,
                          min_dist=min_dist,
                          # metric='cosine',
                          random_state=42).fit_transform(embeddings)

    result = pd.DataFrame(umap_data, columns=['x', 'y'])
    result['labels'] = clusters.labels_

    fig = px.scatter(result, x='x', y='y', color='labels', size_max=20,
                     title='Interactive Clustering Plot',
                     labels={'labels': 'Cluster'})

    fig.update_traces(marker=dict(size=10, opacity=0.7),
                      selector=dict(mode='markers'))
    
    fig.update_layout(width=500, height=500)

    fig.show()

In [ ]:
plot_clusters(embeddings_transcricoes, best_clusters_use)

In [ ]:
plot_clusters_interactive(embeddings_transcricoes, best_clusters_use)

In [ ]:
df["grupo"] = best_clusters_use.labels_
#Salva um excel com os chamados e grupos
#df.to_excel(f"transcricoes_60-180_com_4_clusters_drilldown.xlsx")
df.head()

In [ ]:
df_1 = df[df["grupo"] == 1]
df_0 = df[df["grupo"] == 0]
print(df_1.shape)
print(df_0.shape)

In [ ]:
#Filtrando os grupos distantes
df_filtrada1 = df[df["grupo"] != 1]
df_filtrada1 = df_filtrada1[df_filtrada1["grupo"] != 0]
df_filtrada1.shape

# Segundo Agrupamento

In [ ]:
module_url = "https://tfhub.dev/google/universal-sentence-encoder-multilingual/3"
model_use = hub.load(module_url)
print(f"module {module_url} loaded")
    
todas_transcricoes = list(df_filtrada1['texto_processado_embeddings'])
embeddings_transcricoes = model_use(todas_transcricoes)
print(len(embeddings_transcricoes))

#Definindo o espaço para busca de hiperparâmetros
hspace = {
    "n_neighbors": hp.choice('n_neighbors', range(2, 50)),
    "n_components": hp.choice('n_components', range(2, 50)),
    "min_cluster_size": hp.choice('min_cluster_size', range(2, 12)),
    "min_samples": hp.choice('min_samples', range(2, 12)),
    "random_state": 42
}

label_lower = 20
label_upper = 80
max_evals = 50

best_params_use, best_clusters_use, trials_use = bayesian_search(embeddings_transcricoes,
                                                                 space=hspace,
                                                                 label_lower=label_lower,
                                                                 label_upper=label_upper,
                                                                 max_evals=max_evals)

#Mostrando os melhores hiperparâmetros
print(best_params_use)

In [ ]:
plot_clusters(embeddings_transcricoes, best_clusters_use)

In [ ]:
plot_clusters_interactive(embeddings_transcricoes, best_clusters_use)

In [ ]:
df_filtrada1["grupo"] = best_clusters_use.labels_
#Salva um excel com os chamados e grupos
df_filtrada1.to_excel(f"df_filtrado.xlsx")

In [ ]:
df_filtrada2 = df_filtrada1[df_filtrada1["grupo"] != 0]
print(df_filtrada2.shape)

## Terceiro Agrupamento

In [ ]:
module_url = "https://tfhub.dev/google/universal-sentence-encoder-multilingual/3"
model_use = hub.load(module_url)
print(f"module {module_url} loaded")
    
todas_transcricoes = list(df_filtrada2['texto_processado_embeddings'])
embeddings_transcricoes = model_use(todas_transcricoes)
print(len(embeddings_transcricoes))

#Definindo o espaço para busca de hiperparâmetros
hspace = {
    "n_neighbors": hp.choice('n_neighbors', range(2, 50)),
    "n_components": hp.choice('n_components', range(2, 50)),
    "min_cluster_size": hp.choice('min_cluster_size', range(2, 12)),
    "min_samples": hp.choice('min_samples', range(2, 12)),
    "random_state": 42
}

label_lower = 20
label_upper = 80
max_evals = 50

best_params_use, best_clusters_use, trials_use = bayesian_search(embeddings_transcricoes,
                                                                 space=hspace,
                                                                 label_lower=label_lower,
                                                                 label_upper=label_upper,
                                                                 max_evals=max_evals)

#Mostrando os melhores hiperparâmetros
print(best_params_use)

In [ ]:
clusters = list(np.unique(best_clusters_use.labels_))
len(clusters)

In [ ]:
plot_clusters(embeddings_transcricoes, best_clusters_use)

In [ ]:
plot_clusters_interactive(embeddings_transcricoes, best_clusters_use)

In [ ]:
df_filtrada2["grupo"] = best_clusters_use.labels_
df_filtrada2.head()